# Per-Parent Highlight Sentiment Classifier

Does knowing the **parent** improve sentiment prediction beyond the population-level model?

**Setup**: 20 participants, 4–35 selections each (median ≈ 11).

**Evaluation**: Leave-One-Participant-Out CV (LOPO) — train on 19 participants, test on the 20th.

**Approaches**:
1. Universal structured baseline (Groups A+B+C, 80 features), LOPO-CV
2. Universal + parent history features (Group D-parent), LOPO-CV
3. Parent-conditioned few-shot Claude — training-pool examples + parent demographic profile in prompt

**Batch workflow** (phases 1–4) used for approach 3:
1. Prepare — build requests for all uncached (participant, row, task) combinations
2. Submit — one batch, ID saved to `batch_ids.json`
3. Poll — wait for completion, write to shared cache
4. Evaluate — read from cache, compute per-parent and aggregate metrics

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy as scipy_entropy

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error
import anthropic
from dotenv import dotenv_values

DATA_DIR       = Path('../data-exports/20260412_183830/highlight_analysis_output')
MOD_FILE       = Path('../data-exports/20260412_183830/'
                      'moderation_sessions_export_20260412_183830.csv')
OUT_DIR        = DATA_DIR / 'classifier_output'
OUT_DIR.mkdir(exist_ok=True)

CACHE_FILE     = OUT_DIR / 'llm_predictions_cache.json'
BATCH_IDS_FILE = OUT_DIR / 'batch_ids.json'

RANDOM_STATE   = 42
CLAUDE_MODEL   = 'claude-opus-4-7'
POLL_INTERVAL  = 30

try:
    BEST_K = int(
        pd.read_csv(OUT_DIR / 'cv_fewshot_binary.csv')
        .pipe(lambda d: d.loc[d['AUC (conf)'].idxmax(), 'k'])
    )
except FileNotFoundError:
    BEST_K = 4
    print("cv_fewshot_binary.csv not found — using BEST_K=4.")

print(f"BEST_K: {BEST_K}  |  Model: {CLAUDE_MODEL}")

BEST_K: 8  |  Model: claude-opus-4-7


## Load data

In [2]:
df_sel = pd.read_csv(DATA_DIR / 'df_sel.csv')
mod    = pd.read_csv(MOD_FILE)
scenario_text = mod.drop_duplicates('scenario_id')[['scenario_id', 'scenario_prompt', 'original_response']]
df = df_sel.merge(scenario_text, on='scenario_id', how='left').reset_index(drop=True)

participants = df['prolific_pid'].unique()
print(f"Shape: {df.shape}  |  Participants: {len(participants)}")
print(df['prolific_pid'].value_counts().sort_values().to_string())

Shape: (267, 50)  |  Participants: 20
prolific_pid
67c4c8f9fa9ba2b8a265b2f4     4
5d63867550d0dd0017805976     4
671e55aeca0216c5762de35a     4
623519c893eaebfcb0046e26     5
69c3239a158bfee31783d53f     5
663dff19b292e4e2c5c11a1b     5
69b9174a05fa2db6d0a8b4ff     6
671bbaf1bec15d847b871b9b     7
65c10c990ad597a9eb20043a     8
69c6965bfee6e955699f09e4    10
62b5e825a5e8616bde25741d    11
67294a9141e1aef9d72b7d24    11
67095387c465675aea460ef4    12
610713aeac4904cc5d90e76d    14
664502f636c5bc96a3e056cd    16
673d25851173193daa4771ca    17
69be087681bb16d5cd77fb03    29
66639889ea005547089b6cbf    31
607668ac450fc6e3025a131d    33
69a6dc43a9fe426c65b207a5    35


In [3]:
y_binary     = (df['highlight_sentiment'] >= 5).astype(int).values
y_rating_raw = df['highlight_sentiment'].astype(int).values


def build_structured_features(df_in: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df_in.index)
    out['strategy_is_null'] = df_in['model_strategy'].isna().astype(int)
    out = pd.concat([out,
        pd.get_dummies(df_in['model_strategy'].fillna('none'), prefix='strat'),
        pd.get_dummies(df_in['parent_motivation'].fillna('unknown'), prefix='motiv'),
    ], axis=1)
    for col in ['domain', 'age_band', 'sensitivity_level', 'relationship_frame',
                'space_type', 'trait', 'trait_level']:
        out = pd.concat([out, pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)], axis=1)
    out['breakdown_expected'] = (df_in['breakdown_expected'] == 'yes').astype(int)
    for col in ['parent_gender', 'parent_age_group', 'parent_education', 'parent_ethnicity',
                'area_of_residency', 'child_has_ai_use', 'parent_llm_monitoring_level']:
        out = pd.concat([out, pd.get_dummies(df_in[col].fillna('unknown'), prefix=col)], axis=1)
    out['genai_regular_user'] = (df_in['genai_familiarity'] == 'regular_user').astype(int)
    freq_map = {'daily': 2, 'weekly': 1, 'monthly_or_less': 0}
    out['genai_usage_freq'] = df_in['genai_usage_frequency'].map(freq_map).fillna(0).astype(int)
    out['parent_internet_use_frequency'] = pd.to_numeric(
        df_in['parent_internet_use_frequency'], errors='coerce').fillna(0)
    out['is_only_child'] = (df_in['is_only_child'].astype(str).str.lower() == 'yes').astype(int)
    for s in 'ABCD':
        out[f'parenting_{s}'] = df_in['parenting_style'].fillna('').str.contains(s).astype(int)
    return out.astype(float)


X_struct = build_structured_features(df)
print(f"Structured features: {X_struct.shape[1]}")

Structured features: 82


## Part 1 — Universal model LOPO-CV (structured baseline)

In [4]:
def run_lopo_structured(df, X, y_binary, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask  = (df['prolific_pid'] == pid).values
        X_tr, X_te = X[~test_mask], X[test_mask]
        y_tr, y_te = y_binary[~test_mask], y_binary[test_mask]

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            try:
                y_sc  = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') \
                        else y_pred.astype(float)
                auc   = roc_auc_score(y_te, y_sc) if len(np.unique(y_te)) > 1 else np.nan
            except Exception:
                auc = np.nan
            rows.append({'prolific_pid': pid, 'n_test': test_mask.sum(), 'model': name,
                         'F1':  round(f1_score(y_te, y_pred, zero_division=0), 3),
                         'AUC': round(auc, 3) if not np.isnan(auc) else np.nan,
                         'accuracy': round((y_pred == y_te).mean(), 3)})
    return pd.DataFrame(rows)


def run_lopo_rating(df, X, y_rating, models: dict) -> pd.DataFrame:
    """LOPO-CV for the 1-7 rating task. Reports MAE and Pearson r per participant."""
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask  = (df['prolific_pid'] == pid).values
        X_tr, X_te = X[~test_mask], X[test_mask]
        y_tr, y_te = y_rating[~test_mask], y_rating[test_mask]

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = np.clip(np.round(model.predict(X_te)).astype(int), 1, 7)
            mae    = mean_absolute_error(y_te, y_pred)
            r      = np.corrcoef(y_te, y_pred)[0, 1] if len(y_te) > 1 else np.nan
            rows.append({
                'prolific_pid': pid,
                'n_test':       test_mask.sum(),
                'model':        name,
                'MAE':          round(mae, 3),
                'Pearson_r':    round(r, 3) if not np.isnan(r) else np.nan,
            })
    return pd.DataFrame(rows)


lopo_models = {
    'Majority Baseline':   DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, max_depth=8,
                                                   random_state=RANDOM_STATE),
    'SVM (linear)':        LinearSVC(max_iter=2000, random_state=RANDOM_STATE),
}

rating_models = {
    'Mean Baseline': DummyClassifier(strategy='mean'),  # placeholder — overridden below
    'Ridge':         Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=8,
                                            random_state=RANDOM_STATE),
}

# Mean baseline: predict global training mean (re-implemented as a regressor)
class MeanRegressor:
    def fit(self, X, y):   self.mean_ = y.mean(); return self
    def predict(self, X):  return np.full(len(X), self.mean_)

rating_models['Mean Baseline'] = MeanRegressor()

lopo_universal = run_lopo_structured(df, X_struct.values, y_binary, lopo_models)
print("Universal LOPO-CV — binary (mean across participants):")
print(lopo_universal.groupby('model')[['F1', 'AUC', 'accuracy']].mean().round(3).to_string())
lopo_universal.to_csv(OUT_DIR / 'per_parent_lopo_universal.csv', index=False)

lopo_universal_rating = run_lopo_rating(df, X_struct.values, y_rating_raw, rating_models)
print("\nUniversal LOPO-CV — rating 1–7 (mean across participants):")
print(lopo_universal_rating.groupby('model')[['MAE', 'Pearson_r']].mean().round(3).to_string())
lopo_universal_rating.to_csv(OUT_DIR / 'per_parent_lopo_universal_rating.csv', index=False)

Universal LOPO-CV — binary (mean across participants):
                        F1    AUC  accuracy
model                                      
Logistic Regression  0.549  0.585     0.560
Majority Baseline    0.703  0.500     0.624
Random Forest        0.635  0.697     0.586
SVM (linear)         0.451  0.481     0.510

Universal LOPO-CV — rating 1–7 (mean across participants):
                 MAE  Pearson_r
model                          
Mean Baseline  1.795        NaN
Random Forest  1.644      0.404
Ridge          1.801      0.125


## Part 2 — Universal + parent history features (Group D-parent)

In [5]:
def build_parent_history_features(df_train: pd.DataFrame, n_test: int) -> np.ndarray:
    """Population-level aggregates from training fold, broadcast to n_test rows."""
    mean_sent  = df_train['highlight_sentiment'].mean()
    pct_pos    = (df_train['highlight_sentiment'] >= 5).mean()
    sc         = df_train['model_strategy'].value_counts(normalize=True)
    strat_ent  = scipy_entropy(sc.values) if len(sc) > 1 else 0.0
    return np.tile([mean_sent, pct_pos, strat_ent], (n_test, 1))


def run_lopo_with_history(df, X_struct, y_binary, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask   = (df['prolific_pid'] == pid).values
        df_tr, df_te = df[~test_mask], df[test_mask]
        hist_tr = build_parent_history_features(df_tr, (~test_mask).sum())
        hist_te = build_parent_history_features(df_tr, test_mask.sum())
        X_tr    = np.hstack([X_struct[~test_mask], hist_tr])
        X_te    = np.hstack([X_struct[test_mask],  hist_te])
        y_tr, y_te = y_binary[~test_mask], y_binary[test_mask]

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            try:
                y_sc  = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') \
                        else y_pred.astype(float)
                auc   = roc_auc_score(y_te, y_sc) if len(np.unique(y_te)) > 1 else np.nan
            except Exception:
                auc = np.nan
            rows.append({'prolific_pid': pid, 'n_test': test_mask.sum(), 'model': name,
                         'F1':  round(f1_score(y_te, y_pred, zero_division=0), 3),
                         'AUC': round(auc, 3) if not np.isnan(auc) else np.nan,
                         'accuracy': round((y_pred == y_te).mean(), 3)})
    return pd.DataFrame(rows)


def run_lopo_with_history_rating(df, X_struct, y_rating, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask    = (df['prolific_pid'] == pid).values
        df_tr        = df[~test_mask]
        hist_tr      = build_parent_history_features(df_tr, (~test_mask).sum())
        hist_te      = build_parent_history_features(df_tr, test_mask.sum())
        X_tr         = np.hstack([X_struct[~test_mask], hist_tr])
        X_te         = np.hstack([X_struct[test_mask],  hist_te])
        y_tr, y_te   = y_rating[~test_mask], y_rating[test_mask]

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = np.clip(np.round(model.predict(X_te)).astype(int), 1, 7)
            mae    = mean_absolute_error(y_te, y_pred)
            r      = np.corrcoef(y_te, y_pred)[0, 1] if len(y_te) > 1 else np.nan
            rows.append({
                'prolific_pid': pid,
                'n_test':       test_mask.sum(),
                'model':        name,
                'MAE':          round(mae, 3),
                'Pearson_r':    round(r, 3) if not np.isnan(r) else np.nan,
            })
    return pd.DataFrame(rows)


lopo_history = run_lopo_with_history(df, X_struct.values, y_binary, lopo_models)
print("Universal + population history LOPO-CV — binary (mean):")
print(lopo_history.groupby('model')[['F1', 'AUC', 'accuracy']].mean().round(3).to_string())
lopo_history.to_csv(OUT_DIR / 'per_parent_lopo_history.csv', index=False)

lopo_history_rating = run_lopo_with_history_rating(df, X_struct.values, y_rating_raw, rating_models)
print("\nUniversal + population history LOPO-CV — rating 1–7 (mean):")
print(lopo_history_rating.groupby('model')[['MAE', 'Pearson_r']].mean().round(3).to_string())
lopo_history_rating.to_csv(OUT_DIR / 'per_parent_lopo_history_rating.csv', index=False)

Universal + population history LOPO-CV — binary (mean):
                        F1    AUC  accuracy
model                                      
Logistic Regression  0.549  0.585     0.560
Majority Baseline    0.703  0.500     0.624
Random Forest        0.646  0.608     0.629
SVM (linear)         0.451  0.481     0.510

Universal + population history LOPO-CV — rating 1–7 (mean):
                 MAE  Pearson_r
model                          
Mean Baseline  1.795        NaN
Random Forest  1.672      0.235
Ridge          1.801      0.125


## Part 3 — Universal + per-participant history features

Uses the **test parent's own other selections** as history features (within-participant LOO),
rather than population aggregates. This directly captures individual parent tendencies.

For each test row of participant X, features are computed from X's *other* selections
(excluding the row being predicted). Training participants use all their own training rows.

4 new features (80 → 84 total):
- `parent_mean_sentiment` — mean of their other selections' ratings
- `parent_pct_positive` — fraction rated ≥5
- `parent_strategy_entropy` — Shannon entropy of model strategies they highlighted
- `parent_n_prior` — count of other selections (all participants have ≥3)

In [6]:
def _pp_features_for_rows(df_rows: pd.DataFrame, exclude_idx, pop_fallback: dict) -> np.ndarray:
    """
    For each row in df_rows, compute history features from all OTHER rows with the same
    prolific_pid (within df_rows), excluding the row at position exclude_idx[i].
    exclude_idx is a list/array of df_rows integer positions to exclude (one per row).
    """
    result = []
    for i, (_, row) in enumerate(df_rows.iterrows()):
        pid    = row['prolific_pid']
        others = df_rows[(df_rows['prolific_pid'] == pid) &
                         (df_rows.index != df_rows.index[i])]
        if len(others) == 0:
            result.append([
                pop_fallback['mean_sent'],
                pop_fallback['pct_pos'],
                pop_fallback['strat_ent'],
                0,
            ])
        else:
            sc  = others['model_strategy'].value_counts(normalize=True)
            ent = scipy_entropy(sc.values) if len(sc) > 1 else 0.0
            result.append([
                others['highlight_sentiment'].mean(),
                (others['highlight_sentiment'] >= 5).mean(),
                ent,
                len(others),
            ])
    return np.array(result)


def run_lopo_with_per_participant(df, X_struct, y_binary, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask  = (df['prolific_pid'] == pid).values
        df_tr      = df[~test_mask].reset_index(drop=True)
        df_te      = df[test_mask].reset_index(drop=True)

        pop_fb = {
            'mean_sent': df_tr['highlight_sentiment'].mean(),
            'pct_pos':   (df_tr['highlight_sentiment'] >= 5).mean(),
            'strat_ent': scipy_entropy(
                df_tr['model_strategy'].value_counts(normalize=True).values
            ) if df_tr['model_strategy'].nunique() > 1 else 0.0,
        }

        # Training: each row's history = all same-pid rows in df_tr (no exclusion needed
        # since the label of the row itself is fine for neighbors, not for the row itself)
        pp_tr = _pp_features_for_rows(df_tr, None, pop_fb)
        pp_te = _pp_features_for_rows(df_te, None, pop_fb)

        X_tr = np.hstack([X_struct[~test_mask], pp_tr])
        X_te = np.hstack([X_struct[test_mask],  pp_te])
        y_tr, y_te = y_binary[~test_mask], y_binary[test_mask]

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            try:
                y_sc  = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') \
                        else y_pred.astype(float)
                auc   = roc_auc_score(y_te, y_sc) if len(np.unique(y_te)) > 1 else np.nan
            except Exception:
                auc = np.nan
            rows.append({'prolific_pid': pid, 'n_test': test_mask.sum(), 'model': name,
                         'F1':  round(f1_score(y_te, y_pred, zero_division=0), 3),
                         'AUC': round(auc, 3) if not np.isnan(auc) else np.nan,
                         'accuracy': round((y_pred == y_te).mean(), 3)})
    return pd.DataFrame(rows)


def run_lopo_with_per_participant_rating(df, X_struct, y_rating, models: dict) -> pd.DataFrame:
    rows = []
    for pid in df['prolific_pid'].unique():
        test_mask  = (df['prolific_pid'] == pid).values
        df_tr      = df[~test_mask].reset_index(drop=True)
        df_te      = df[test_mask].reset_index(drop=True)

        pop_fb = {
            'mean_sent': df_tr['highlight_sentiment'].mean(),
            'pct_pos':   (df_tr['highlight_sentiment'] >= 5).mean(),
            'strat_ent': scipy_entropy(
                df_tr['model_strategy'].value_counts(normalize=True).values
            ) if df_tr['model_strategy'].nunique() > 1 else 0.0,
        }

        pp_tr = _pp_features_for_rows(df_tr, None, pop_fb)
        pp_te = _pp_features_for_rows(df_te, None, pop_fb)

        X_tr = np.hstack([X_struct[~test_mask], pp_tr])
        X_te = np.hstack([X_struct[test_mask],  pp_te])
        y_tr, y_te = y_rating[~test_mask], y_rating[test_mask]

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            y_pred = np.clip(np.round(model.predict(X_te)).astype(int), 1, 7)
            mae    = mean_absolute_error(y_te, y_pred)
            r      = np.corrcoef(y_te, y_pred)[0, 1] if len(y_te) > 1 else np.nan
            rows.append({
                'prolific_pid': pid,
                'n_test':       test_mask.sum(),
                'model':        name,
                'MAE':          round(mae, 3),
                'Pearson_r':    round(r, 3) if not np.isnan(r) else np.nan,
            })
    return pd.DataFrame(rows)


lopo_pp = run_lopo_with_per_participant(df, X_struct.values, y_binary, lopo_models)
print("Universal + per-participant history LOPO-CV — binary (mean):")
print(lopo_pp.groupby('model')[['F1', 'AUC', 'accuracy']].mean().round(3).to_string())
lopo_pp.to_csv(OUT_DIR / 'per_parent_lopo_per_participant.csv', index=False)

lopo_pp_rating = run_lopo_with_per_participant_rating(df, X_struct.values, y_rating_raw, rating_models)
print("\nUniversal + per-participant history LOPO-CV — rating 1–7 (mean):")
print(lopo_pp_rating.groupby('model')[['MAE', 'Pearson_r']].mean().round(3).to_string())
lopo_pp_rating.to_csv(OUT_DIR / 'per_parent_lopo_per_participant_rating.csv', index=False)

# 3-way rating comparison: base | +population | +per-participant
base = lopo_universal_rating.groupby('model')[['MAE', 'Pearson_r']].mean()
pop  = lopo_history_rating.groupby('model')[['MAE', 'Pearson_r']].mean()
pp   = lopo_pp_rating.groupby('model')[['MAE', 'Pearson_r']].mean()
cmp3 = base.join(pop, lsuffix='_base', rsuffix='_pop').join(
    pp.rename(columns={'MAE': 'MAE_pp', 'Pearson_r': 'r_pp'})
)
cmp3.columns = ['MAE', 'r', 'MAE (+pop)', 'r (+pop)', 'MAE (+pp)', 'r (+pp)']
cmp3 = cmp3.round(3)
print("\nRating task — 3-way comparison (base | +population history | +per-participant history):")
print(cmp3.to_string())
cmp3.to_csv(OUT_DIR / 'per_parent_rating_comparison.csv')

# Verify: check that per-participant features vary across test rows for a sample participant
sample_pid = df['prolific_pid'].value_counts().idxmax()
sample_rows = df[df['prolific_pid'] == sample_pid].reset_index(drop=True)
pop_fb_check = {'mean_sent': df['highlight_sentiment'].mean(),
                'pct_pos': (df['highlight_sentiment'] >= 5).mean(), 'strat_ent': 0.0}
pp_sample = _pp_features_for_rows(sample_rows, None, pop_fb_check)
n_unique = len(set(tuple(r) for r in pp_sample))
print(f"\nVerification — participant {sample_pid[:8]} ({len(sample_rows)} rows): "
      f"{n_unique} unique feature vectors (expected = {len(sample_rows)})")

Universal + per-participant history LOPO-CV — binary (mean):
                        F1    AUC  accuracy
model                                      
Logistic Regression  0.581  0.515     0.642
Majority Baseline    0.703  0.500     0.624
Random Forest        0.598  0.490     0.607
SVM (linear)         0.433  0.488     0.467

Universal + per-participant history LOPO-CV — rating 1–7 (mean):
                 MAE  Pearson_r
model                          
Mean Baseline  1.795        NaN
Random Forest  1.722     -0.192
Ridge          2.549      0.288

Rating task — 3-way comparison (base | +population history | +per-participant history):
                 MAE      r  MAE (+pop)  r (+pop)  MAE (+pp)  r (+pp)
model                                                                
Mean Baseline  1.795    NaN       1.795       NaN      1.795      NaN
Random Forest  1.644  0.404       1.672     0.235      1.722   -0.192
Ridge          1.801  0.125       1.801     0.125      2.549    0.288

Verificat

## Part 3 — Parent-conditioned few-shot Claude (batch)

In LOPO the held-out parent has zero training rows, so "parent-conditioned" means:
- Few-shot examples from the 19-participant training pool (stratified by class)
- Held-out parent's **demographic profile** injected as a prompt header

In [ ]:
env      = dotenv_values(Path('.') / '.env')
api_key  = env.get('CLAUDE_API_KEY')
base_url = env.get('CLAUDE_API_BASE_URL')

SKIP_CLAUDE = not api_key
if not SKIP_CLAUDE:
    client = anthropic.Anthropic(api_key=api_key, base_url=base_url)
    print(f"Client ready. Model: {CLAUDE_MODEL}")
else:
    print("WARNING: CLAUDE_API_KEY not found — skipping Claude cells.")


def load_cache() -> dict:
    return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

def save_cache(c: dict) -> None:
    CACHE_FILE.write_text(json.dumps(c, indent=2))

def load_batch_ids() -> dict:
    return json.loads(BATCH_IDS_FILE.read_text()) if BATCH_IDS_FILE.exists() else {}

def save_batch_ids(b: dict) -> None:
    BATCH_IDS_FILE.write_text(json.dumps(b, indent=2))

In [ ]:
SYSTEM_BINARY_PARENT = """You are an expert evaluator of AI responses to children's questions.
A parent reviewed an AI chatbot's response to their child's message and highlighted
a specific passage. They then rated that passage on a 1–7 sentiment scale where
1 = very negative or inappropriate and 7 = very positive or appropriate.

Your task is to predict whether this specific parent rated the highlighted passage as
POSITIVE (rating ≥5) or NEGATIVE/NEUTRAL (rating < 5).

You will be given the parent's demographic profile, labeled examples from other parents,
and the case to predict.

Respond ONLY with a JSON object in this exact format, no other text:
{"prediction": 0 or 1, "confidence": 0.0 to 1.0}"""

SYSTEM_RATING_PARENT = """You are an expert evaluator of AI responses to children's questions.
A parent reviewed an AI chatbot's response to their child's message and highlighted
a specific passage. They then rated that passage on a 1–7 sentiment scale where
1 = very negative or inappropriate and 7 = very positive or appropriate.

Your task is to predict this specific parent's 1–7 sentiment rating.
You will be given the parent's demographic profile, labeled examples from other parents,
and the case to predict.

Respond ONLY with a JSON object in this exact format, no other text:
{"rating": <integer 1-7>}"""


def make_case_block(row: pd.Series) -> str:
    return (f"AGE BAND: {row.get('age_band', 'unknown')}\n"
            f"DOMAIN: {row.get('domain', 'unknown')}\n\n"
            f"CHILD'S QUESTION:\n{row['scenario_prompt']}\n\n"
            f"FULL AI RESPONSE:\n{row['original_response']}\n\n"
            f"HIGHLIGHTED TEXT:\n{row['highlight_text']}")


def make_parent_profile(row: pd.Series) -> str:
    return (f"PARENT PROFILE:\n"
            f"  Gender: {row.get('parent_gender', 'unknown')}\n"
            f"  Age group: {row.get('parent_age_group', 'unknown')}\n"
            f"  Education: {row.get('parent_education', 'unknown')}\n"
            f"  Area: {row.get('area_of_residency', 'unknown')}\n"
            f"  GenAI familiarity: {row.get('genai_familiarity', 'unknown')}\n"
            f"  LLM monitoring: {row.get('parent_llm_monitoring_level', 'unknown')}\n"
            f"  Parenting style: {row.get('parenting_style', 'unknown')}")


def make_fewshot_prefix(df_train: pd.DataFrame, k: int, random_state: int = 42) -> str:
    neg   = df_train[df_train['highlight_sentiment'] < 5]
    pos   = df_train[df_train['highlight_sentiment'] >= 5]
    k_neg, k_pos = k // 2, k - k // 2
    examples = pd.concat([
        neg.sample(n=k_neg, replace=len(neg) < k_neg, random_state=random_state),
        pos.sample(n=k_pos, replace=len(pos) < k_pos, random_state=random_state),
    ]).sample(frac=1, random_state=random_state)

    parts = ["EXAMPLES FROM OTHER PARENTS (calibrate your predictions using these):\n"]
    for i, (_, row) in enumerate(examples.iterrows(), 1):
        rating = int(row['highlight_sentiment'])
        label  = 'POSITIVE (≥5)' if rating >= 5 else 'NEGATIVE/NEUTRAL (<5)'
        parts.append(f"[EXAMPLE {i} — TRUE RATING: {rating} ({label})]\n" + make_case_block(row))
    parts.append("\nNow predict for the following parent:")
    return "\n\n".join(parts)

### Phase 1 — Prepare

In [ ]:
def prepare_parent_requests(df: pd.DataFrame, k: int, cache: dict) -> list[dict]:
    requests = []
    pids     = df['prolific_pid'].unique()

    for pid_idx, pid in enumerate(pids):
        test_mask = (df['prolific_pid'] == pid).values
        df_train  = df[~test_mask]
        df_test   = df[test_mask]
        test_pos  = np.where(test_mask)[0]

        prefix  = make_fewshot_prefix(df_train, k, random_state=pid_idx * 100 + k)
        profile = make_parent_profile(df_test.iloc[0])

        for pos, (_, row) in zip(test_pos, df_test.iterrows()):
            user_msg = prefix + "\n\n" + profile + "\n\n" + make_case_block(row)

            for task, system in [('binary', SYSTEM_BINARY_PARENT),
                                  ('rating', SYSTEM_RATING_PARENT)]:
                key = f"parent_k{k}_pid{pid_idx}_{task}_pos{pos}"
                if key not in cache:
                    requests.append({
                        "custom_id": key,
                        "params": {
                            "model": CLAUDE_MODEL, "max_tokens": 64,
                            "system": [{"type": "text", "text": system,
                                        "cache_control": {"type": "ephemeral"}}],
                            "messages": [{"role": "user", "content": user_msg}],
                        },
                    })
    return requests


if not SKIP_CLAUDE:
    cache    = load_cache()
    requests = prepare_parent_requests(df, BEST_K, cache)
    print(f"Requests to submit: {len(requests)}  "
          f"(cached: {len(df) * 2 - len(requests)})")

### Phase 2 — Submit

In [ ]:
if not SKIP_CLAUDE and requests:
    batch      = client.messages.batches.create(requests=requests)
    batch_id   = batch.id
    batch_ids  = load_batch_ids()
    batch_ids['per_parent'] = batch_id
    save_batch_ids(batch_ids)
    print(f"Batch submitted: {batch_id}  ({len(requests)} requests)")
elif not SKIP_CLAUDE:
    print("All requests cached — skipping submission.")
    batch_id = load_batch_ids().get('per_parent')

### Phase 3 — Poll and save

In [ ]:
if not SKIP_CLAUDE:
    if 'batch_id' not in dir() or batch_id is None:
        batch_id = load_batch_ids().get('per_parent')
        if not batch_id:
            raise RuntimeError("No batch_id found — run the submit cell first.")
        print(f"Recovered batch_id: {batch_id}")

    print(f"Polling {batch_id} every {POLL_INTERVAL}s…")
    while True:
        batch  = client.messages.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f"  {batch.processing_status} — "
              f"processing={counts.processing}  succeeded={counts.succeeded}  "
              f"errored={counts.errored}")
        if batch.processing_status == 'ended':
            break
        time.sleep(POLL_INTERVAL)

    cache    = load_cache()
    n_ok, n_err = 0, 0
    for result in client.messages.batches.results(batch_id):
        if result.result.type == 'succeeded':
            try:
                cache[result.custom_id] = json.loads(
                    result.result.message.content[0].text.strip()
                )
                n_ok += 1
            except Exception:
                cache[result.custom_id] = None
                n_err += 1
        else:
            cache[result.custom_id] = None
            n_err += 1

    save_cache(cache)
    print(f"Cache updated: {n_ok} succeeded, {n_err} failed.")

### Phase 4 — Extract predictions and evaluate

In [ ]:
if not SKIP_CLAUDE:
    cache        = load_cache()
    pids         = df['prolific_pid'].unique()
    binary_preds = np.ones(len(df), dtype=int)
    binary_conf  = np.full(len(df), 0.5)
    rating_preds = np.full(len(df), 5, dtype=int)

    for pid_idx, pid in enumerate(pids):
        test_mask = (df['prolific_pid'] == pid).values
        for pos in np.where(test_mask)[0]:
            b = cache.get(f"parent_k{BEST_K}_pid{pid_idx}_binary_pos{pos}")
            if b and 'prediction' in b:
                binary_preds[pos] = int(b['prediction'])
                binary_conf[pos]  = float(b.get('confidence', 0.5))
            r = cache.get(f"parent_k{BEST_K}_pid{pid_idx}_rating_pos{pos}")
            if r and 'rating' in r:
                rating_preds[pos] = int(np.clip(r['rating'], 1, 7))

    # Per-participant summary
    parent_rows = []
    for pid in pids:
        mask   = (df['prolific_pid'] == pid).values
        y_true = y_binary[mask]
        bp, bc = binary_preds[mask], binary_conf[mask]
        try:
            auc = roc_auc_score(y_true, bc) if len(np.unique(y_true)) > 1 else np.nan
        except Exception:
            auc = np.nan
        parent_rows.append({
            'prolific_pid':  pid,
            'n_selections':  mask.sum(),
            'true_pct_pos':  round(y_true.mean(), 3),
            'pred_pct_pos':  round(bp.mean(), 3),
            'F1_fewshot':    round(f1_score(y_true, bp, zero_division=0), 3),
            'AUC_fewshot':   round(auc, 3) if not np.isnan(auc) else np.nan,
        })

    # Merge with universal RF LOPO
    univ = (lopo_universal[lopo_universal['model'] == 'Random Forest']
            [['prolific_pid', 'F1', 'AUC']]
            .rename(columns={'F1': 'F1_universal', 'AUC': 'AUC_universal'}))
    parent_summary = pd.DataFrame(parent_rows).merge(univ, on='prolific_pid', how='left')
    parent_summary['AUC_delta'] = parent_summary['AUC_fewshot'] - parent_summary['AUC_universal']

    print(parent_summary.sort_values('n_selections').to_string(index=False))
    parent_summary.to_csv(OUT_DIR / 'per_parent_summary.csv', index=False)

    # Aggregate
    agg = pd.DataFrame([
        {'approach': 'Universal RF',            'mean AUC': lopo_universal[lopo_universal['model']=='Random Forest']['AUC'].mean(),
                                                 'mean F1':  lopo_universal[lopo_universal['model']=='Random Forest']['F1'].mean()},
        {'approach': 'Universal + history RF',  'mean AUC': lopo_history[lopo_history['model']=='Random Forest']['AUC'].mean(),
                                                 'mean F1':  lopo_history[lopo_history['model']=='Random Forest']['F1'].mean()},
        {'approach': f'Few-shot k={BEST_K}',    'mean AUC': parent_summary['AUC_fewshot'].mean(),
                                                 'mean F1':  parent_summary['F1_fewshot'].mean()},
    ]).round(3)
    print("\nAggregate comparison:")
    print(agg.to_string(index=False))
    agg.to_csv(OUT_DIR / 'per_parent_aggregate.csv', index=False)

## Visualizations

In [ ]:
if not SKIP_CLAUDE:
    ps    = parent_summary.sort_values('n_selections').reset_index(drop=True)
    valid = ps.dropna(subset=['AUC_universal', 'AUC_fewshot'])
    x, w  = np.arange(len(valid)), 0.35

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - w/2, valid['AUC_universal'], w, label='Universal RF',       color='steelblue')
    ax.bar(x + w/2, valid['AUC_fewshot'],   w, label=f'Few-shot k={BEST_K}', color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{pid[:8]} (n={n})" for pid, n in zip(valid['prolific_pid'], valid['n_selections'])],
        rotation=45, ha='right', fontsize=8,
    )
    ax.set_ylabel('AUC')
    ax.set_title('Per-parent AUC: Universal RF vs. Parent-conditioned Few-Shot Claude')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'per_parent_auc_comparison.png', dpi=100)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(valid['n_selections'], valid['AUC_delta'], color='steelblue', alpha=0.7, s=80)
    for _, r in valid.iterrows():
        ax.annotate(r['prolific_pid'][:6], (r['n_selections'], r['AUC_delta']),
                    fontsize=7, ha='left', va='bottom')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel('n selections per parent')
    ax.set_ylabel('AUC delta (few-shot − universal)')
    ax.set_title('Few-shot improvement vs. sample size')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'per_parent_delta_vs_n.png', dpi=100)
    plt.show()